In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter

In [2]:
data = [
    ("这个 电影 太 好看 了", 1),
    ("演员 演技 在线 剧情 精彩", 1),
    ("非常 喜欢 这部 作品", 1),
    ("画面 精美 值得 一看", 1),
    ("太 浪费 时间 了 烂片", 0),
    ("剧情 拖沓 演技 尴尬", 0),
    ("完全 看 不 下去", 0),
    ("浪费 钱 不 推荐", 0),
]

In [3]:
data

[('这个 电影 太 好看 了', 1),
 ('演员 演技 在线 剧情 精彩', 1),
 ('非常 喜欢 这部 作品', 1),
 ('画面 精美 值得 一看', 1),
 ('太 浪费 时间 了 烂片', 0),
 ('剧情 拖沓 演技 尴尬', 0),
 ('完全 看 不 下去', 0),
 ('浪费 钱 不 推荐', 0)]

In [4]:
counter = Counter()

In [5]:
counter

Counter()

In [6]:
tokenized_data = []

for text, label in data:
    tokens = text.split()
    tokenized_data.append((tokens, label))
    counter.update(tokens)

In [7]:
tokenized_data

[(['这个', '电影', '太', '好看', '了'], 1),
 (['演员', '演技', '在线', '剧情', '精彩'], 1),
 (['非常', '喜欢', '这部', '作品'], 1),
 (['画面', '精美', '值得', '一看'], 1),
 (['太', '浪费', '时间', '了', '烂片'], 0),
 (['剧情', '拖沓', '演技', '尴尬'], 0),
 (['完全', '看', '不', '下去'], 0),
 (['浪费', '钱', '不', '推荐'], 0)]

In [8]:
word2idx = {"<pad>": 0, "<unk>": 1}

In [9]:
word2idx

{'<pad>': 0, '<unk>': 1}

In [10]:
for word, _ in counter.most_common():
    word2idx[word] = len(word2idx)

In [11]:
word2idx

{'<pad>': 0,
 '<unk>': 1,
 '太': 2,
 '了': 3,
 '演技': 4,
 '剧情': 5,
 '浪费': 6,
 '不': 7,
 '这个': 8,
 '电影': 9,
 '好看': 10,
 '演员': 11,
 '在线': 12,
 '精彩': 13,
 '非常': 14,
 '喜欢': 15,
 '这部': 16,
 '作品': 17,
 '画面': 18,
 '精美': 19,
 '值得': 20,
 '一看': 21,
 '时间': 22,
 '烂片': 23,
 '拖沓': 24,
 '尴尬': 25,
 '完全': 26,
 '看': 27,
 '下去': 28,
 '钱': 29,
 '推荐': 30}

In [12]:
class SenimentDataset(Dataset):
    def __init__(self, data, word2idx):
        self.data = data
        self.word2idx = word2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]
        indices = [self.word2idx.get(t, 1) for t in tokens]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [13]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels

In [14]:
dataset = SenimentDataset(tokenized_data, word2idx)

In [15]:
dataset

In [16]:
loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [17]:
loader

In [18]:
for texts, labels in loader:
    print(f"文本形状: {texts.shape}")
    print(f"标签: {labels}")
    break

文本形状: torch.Size([4, 5])
标签: tensor([1, 0, 1, 0])
